In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {'Baseline_Training', 'CTC_models', 'PhoneticFeatures_Training'}:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from project_paths import (
    BASELINE_MODELS_DIR,
    CTC_MODELS_DIR,
    DEFAULT_BASELINE_MODEL,
    DEFAULT_CTC_ATTENTION_MODEL,
    DEFAULT_CTC_MODEL,
    DEFAULT_PHONETIC_MODEL_2CLASS,
    DEFAULT_PHONETIC_MODEL_3CLASS,
    EXAMPLE_EMBEDDINGS,
    EXAMPLE_PHONEMES,
    EXAMPLE_SEG_B2,
    EXAMPLE_SEG_B4,
    EXAMPLE_WAV,
    PHONEME_CHOICES_SUMMARY,
    PHONETIC_MODELS_DIR,
    TABLES_DIR,
)


# Tacotron-style Attention Alignment

This notebook prepares phoneme datasets, trains an attention-based alignment model,
and visualizes phoneme-to-frame alignments. It focuses on the Tacotron-style
experiment and omits old CTC-only debugging cells.


In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONS']="max_split_size:128"
import random

import matplotlib.pyplot as plt
import torch
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter

from CTC_tools import (
    AttentionPhonemeModel,
    CTCPhonemeDataset,
    GuidedAttentionLoss,
    attention_collate_fn,
    build_decoder_inputs,
    load_selected_embeddings_ctc,
    train_attention_model,
)


## Configuration

Update the dataset paths or speaker splits here before running the notebook.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

root_dir = r"\\nid-tts-02\mnt\hot_store\trainee_common\ananeva\CORPRES_embs_no_averaging"
save_path = str(CTC_MODELS_DIR)

uid = "".join(chr(code) for code in [random.randint(65, 90) for _ in range(10)])
result_dir = os.path.join(save_path, uid)
os.makedirs(result_dir, exist_ok=True)

phoneme_list_full = [
    "a0", "a1", "a2", "a4", "b", "b'", "c", "ch", "ch_", "d", "d'",
    "e0", "e1", "e2", "e4", "f", "f'", "g", "g'", "h", "h'", "i0", "i1",
    "i2", "i4", "j", "jr", "jl", "ji4", "k", "k'", "l", "l'", "m", "m'",
    "n", "n'", "o0", "o1", "o2", "o4", "p", "p'", "r", "r'", "s", "s'",
    "sc", "sh", "t", "t'", "u0", "u1", "u2", "u4", "v", "v'", "y0", "y1",
    "y2", "y4", "z", "z'", "zh", "zh'", "C", "CH", "H", "SC",
]
phoneme_list = phoneme_list_full + ["sil"]

train_speakers = ["Vladimir", "Maria", "Mikhail", "Anna"]
test_speakers = ["Galina", "Victoria", "Petr", "Alexander"]

batch_size = 16
num_epochs = 20
hidden_dim = 256
lambda_att = 0.3


## Data Preparation

Load embeddings, build datasets, and verify the shapes produced by the attention
collate function.


In [ ]:
train_sequences, train_targets, train_info = load_selected_embeddings_ctc(
    root_dir=root_dir,
    phoneme_list=phoneme_list,
    speakers=train_speakers,
    max_sequences_per_speaker=1000,
)

test_sequences, test_targets, test_info = load_selected_embeddings_ctc(
    root_dir=root_dir,
    phoneme_list=phoneme_list,
    speakers=test_speakers,
    max_sequences_per_speaker=1000,
)

train_dataset = CTCPhonemeDataset(train_sequences, train_targets, phoneme_list)
test_dataset = CTCPhonemeDataset(test_sequences, test_targets, phoneme_list)

attention_train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=attention_collate_fn,
)

attention_test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=attention_collate_fn,
)

print(f"Train samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Device: {device}")
print(f"Result directory: {result_dir}")


In [ ]:
x_att, y_att, x_lens_att, y_lens_att = next(iter(attention_train_loader))

print("Acoustic batch shape:", x_att.shape)   # [B, T_max, D]
print("Target batch shape:", y_att.shape)     # flattened or padded target representation
print("Input lengths:", x_lens_att[:8])
print("Target lengths:", y_lens_att[:8])


## Model Training

Train the attention model with Guided Attention Loss to encourage monotonic alignments.


In [ ]:
input_dim = train_sequences[0].shape[1]
num_phonemes = len(phoneme_list)

attention_model = AttentionPhonemeModel(
    input_dim=input_dim,
    hidden_dim=hidden_dim,
    num_phonemes=num_phonemes,
    num_layers=2,
    phoneme_emb_dim=128,
    decoder_dim=256,
    attention_dim=128,
).to(device)

attention_optimizer = torch.optim.Adam(attention_model.parameters(), lr=1e-3)
attention_scheduler = torch.optim.lr_scheduler.StepLR(
    attention_optimizer,
    step_size=10,
    gamma=0.5,
)
guided_attention = GuidedAttentionLoss(g=0.2)

best_attention_model_path = os.path.join(result_dir, "best_attention_model.pth")


In [ ]:
writer = SummaryWriter(log_dir=result_dir)
torch.cuda.empty_cache()

train_attention_model(
    model=attention_model,
    train_loader=attention_train_loader,
    test_loader=attention_test_loader,
    optimizer=attention_optimizer,
    scheduler=attention_scheduler,
    num_epochs=num_epochs,
    device=device,
    writer=writer,
    save_path=best_attention_model_path,
    lambda_att=lambda_att,
    guided_loss_fn=guided_attention,
)

writer.close()
print("Saved model to:", best_attention_model_path)


## Alignment Inspection

Helper functions below extract frame boundaries from the attention map and visualize
the alignment for a sample from the test split.


In [ ]:
def phoneme_boundaries_from_attention(
    attention,
    input_lengths=None,
    target_lengths=None,
    method="argmax",
    frame_ms=20.0,
):
    if attention.dim() != 3:
        raise ValueError("attention must have shape [B, L, T]")

    _, max_target_len, max_input_len = attention.shape
    frame_positions = torch.arange(
        max_input_len,
        device=attention.device,
        dtype=attention.dtype,
    )
    boundaries = []

    for batch_index in range(attention.size(0)):
        target_len = target_lengths[batch_index].item() if target_lengths is not None else max_target_len
        input_len = input_lengths[batch_index].item() if input_lengths is not None else max_input_len
        attn = attention[batch_index, :target_len, :input_len]

        if method == "argmax":
            frame_idx = attn.argmax(dim=-1).float()
        elif method == "center_of_mass":
            weights = attn / attn.sum(dim=-1, keepdim=True).clamp_min(1e-8)
            frame_idx = (weights * frame_positions[:input_len]).sum(dim=-1)
        else:
            raise ValueError("method must be 'argmax' or 'center_of_mass'")

        boundaries.append(
            {
                "frame_index": frame_idx.detach().cpu(),
                "time_ms": (frame_idx * frame_ms).detach().cpu(),
            }
        )

    return boundaries


def plot_attention_map(
    attention,
    phoneme_tokens=None,
    input_length=None,
    target_length=None,
    title="Attention alignment",
):
    if attention.dim() != 2:
        raise ValueError("attention must have shape [L, T] for plotting")

    target_len = target_length if target_length is not None else attention.size(0)
    input_len = input_length if input_length is not None else attention.size(1)
    attn = attention[:target_len, :input_len].detach().cpu().numpy()

    plt.figure(figsize=(12, 5))
    plt.imshow(attn, aspect="auto", origin="lower", interpolation="nearest")
    plt.colorbar(label="attention weight")
    plt.xlabel("Frame index (20 ms per frame)")
    plt.ylabel("Phoneme step")
    plt.title(title)
    if phoneme_tokens is not None:
        plt.yticks(range(len(phoneme_tokens)), phoneme_tokens)
    plt.tight_layout()
    plt.show()


In [ ]:
attention_model.eval()

with torch.no_grad():
    x, y, x_lens, y_lens = next(iter(attention_test_loader))
    x = x.to(device)
    y = y.to(device)
    x_lens = x_lens.to(device)
    y_lens = y_lens.to(device)

    decoder_inputs = build_decoder_inputs(y).to(device)
    logits, stop_logits, attention, _ = attention_model(x, decoder_inputs, x_lens)
    pred_ids = logits.argmax(dim=-1)
    stop_probs = torch.sigmoid(stop_logits)

sample_idx = 0
sample_len = y_lens[sample_idx].item()
frame_len = x_lens[sample_idx].item()

true_ids = y[sample_idx, :sample_len].tolist()
pred_ids_sample = pred_ids[sample_idx, :sample_len].detach().cpu().tolist()

true_phonemes = [train_dataset.idx2phoneme[idx] for idx in true_ids]
pred_phonemes = [train_dataset.idx2phoneme.get(idx, "<blank>") for idx in pred_ids_sample]

print("TRUE:", true_phonemes)
print("PRED:", pred_phonemes)
print("STOP probabilities:", stop_probs[sample_idx, :sample_len].detach().cpu())

plot_attention_map(
    attention[sample_idx],
    phoneme_tokens=true_phonemes,
    input_length=frame_len,
    target_length=sample_len,
    title="Tacotron-style phoneme alignment",
)

argmax_boundaries = phoneme_boundaries_from_attention(
    attention,
    input_lengths=x_lens,
    target_lengths=y_lens,
    method="argmax",
)
center_boundaries = phoneme_boundaries_from_attention(
    attention,
    input_lengths=x_lens,
    target_lengths=y_lens,
    method="center_of_mass",
)

print("Argmax boundaries (ms):", argmax_boundaries[sample_idx]["time_ms"])
print("Center-of-mass boundaries (ms):", center_boundaries[sample_idx]["time_ms"])


In [ ]:
len(argmax_boundaries[sample_idx]['time_ms'])

In [ ]:
for i, ph in enumerate(pred_phonemes):
    print(f'{int(argmax_boundaries[sample_idx]["time_ms"][i]*2*1600)}, 2, {ph}')